In [1]:
import os
os.chdir('/')
os.chdir('/home/sudip/ml_projects/song_sug')
os.getcwd()

'/home/sudip/ml_projects/song_sug'

In [2]:
import pandas as pd
df = pd.read_csv('data/raw/training.csv')
df.head(3)

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3


In [3]:
import spacy 
nlp = spacy.load("en_core_web_sm")


def process_text(text_list):
   for docs in text_list:
        doc = nlp(docs)
        tokens = [
            token.text for token in doc
            ]
        yield tokens

training_sequences = df['text']

training_token = list(process_text(training_sequences))

# CBOW

working better then other two SG and the noSG|CBOW

In [4]:
from gensim.models import Word2Vec

model_cbow = Word2Vec(training_token, vector_size=100, window=7, min_count=3, sg=0, epochs=30)

# print("CBOW:", model_cbow.wv.most_similar('guilt', topn=10))
print(model_cbow)

Word2Vec<vocab=5224, vector_size=100, alpha=0.025>


# Using GRU

Prepare the dataset 

In [5]:
import numpy as np 

def sentence_to_vec( Sentance_tokens, w2v_model, max_len = 40, embding_dim = 100):

    vecs = []

    for word in Sentance_tokens:

        if word in w2v_model.wv:
            vecs.append(w2v_model.wv[word])
        else:
            vecs.append(np.zeros(embding_dim))
    
    if len(vecs) < max_len:
        vecs += [np.zeros(embding_dim)] * (max_len - len(vecs))
    else:
        vecs = vecs[:max_len]
    
    return np.array(vecs)

In [6]:
X_train = np.array([sentence_to_vec(token, model_cbow) for token in training_token])
y_train = df['label'].values

In [7]:
print(np.max(np.abs(X_train)))
print('Y_train:', np.unique(y_train)[:10])
print(X_train.shape)
print(y_train.shape)

6.462850570678711
Y_train: [0 1 2 3 4 5]
(16000, 40, 100)
(16000,)


build GRU model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

num_classes = 6


model = Sequential([
        GRU(units=64, return_sequences=True, input_shape=(40, 100)),
        Dropout(0.5),
        GRU(units=64),
        Dense(num_classes, activation='softmax')
        ])



2026-02-10 14:34:16.826656: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1770734061.732077    1002 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2246 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5
/home/sudip/ml_projects/song_sug/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
model.compile(
    optimizer=Adam(learning_rate=1e-3, clipnorm=1.0), 
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
    )

In [10]:
import pandas as pd
val_df = pd.read_csv('data/raw/validation.csv')
val_df.head(3)

,text,label
0,im feeling quite sad and sorry for myself but ...,0
1,i feel like i am still looking at a blank canv...,0
2,i feel like a faithful servant,2


In [11]:
X_val = np.array([sentence_to_vec(token, model_cbow) for token in 
                  list(process_text(val_df['text']))])
y_val = val_df['label'].values

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

In [13]:
history = model.fit(
    X_train , y_train,
    epochs = 20,
    batch_size = 32,
    
    validation_data = (X_val, y_val),
    callbacks = [early_stopping]
    )

2026-02-10 14:34:37.533414: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 256000000 exceeds 10% of free system memory.
2026-02-10 14:34:38.719717: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 256000000 exceeds 10% of free system memory.


Epoch 1/20


2026-02-10 14:34:41.182741: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3372 - loss: 1.6051

2026-02-10 14:34:52.730521: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 32000000 exceeds 10% of free system memory.
2026-02-10 14:34:52.775292: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 32000000 exceeds 10% of free system memory.


500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.3309 - loss: 1.5910 - val_accuracy: 0.3480 - val_loss: 1.5855
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.3350 - loss: 1.5718 - val_accuracy: 0.3480 - val_loss: 1.5828
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.4636 - loss: 1.3912 - val_accuracy: 0.5695 - val_loss: 1.1382
Epoch 4/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.6826 - loss: 0.8266 - val_accuracy: 0.7695 - val_loss: 0.6056
Epoch 5/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.8271 - loss: 0.4731 - val_accuracy: 0.8520 - val_loss: 0.4244
Epoch 6/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.8815 - loss: 0.3285 - val_accuracy: 0.8805 - val_loss: 0.3166
Epoch 7/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.9062 - loss: 0.2481 - val_accuracy: 0.8975 - val_loss: 0.2550
Epoch 8/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9171 - loss: 0.2076 - val_accurac

# TESTING

In [14]:
import pandas as pd
text_df = pd.read_csv('data/raw/test.csv')
text_df.head(3)

,text,label
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0


In [15]:
X_test = np.array([sentence_to_vec(token, model_cbow) for token in 
                   list(process_text(text_df['text']))])
y_test = text_df['label'].values

In [16]:
print(X_test.shape, '\n',
        y_test.shape)

(2000, 40, 100) 
 (2000,)


In [17]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, batch_size=32)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


10/63 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9286 - loss: 0.1828

2026-02-10 14:37:09.805590: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 32000000 exceeds 10% of free system memory.


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9010 - loss: 0.2602
Test Loss: 0.26021459698677063
Test Accuracy: 0.9010000228881836


In [18]:
predictions = model.predict(X_test)

print(predictions[:5])


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
[[9.9922287e-01 2.3854876e-04 7.1557872e-05 3.2513405e-04 1.3994020e-04
  1.9303420e-06]
 [9.9906915e-01 2.3775443e-04 5.2756754e-05 5.2856520e-04 1.0972752e-04
  2.0562552e-06]
 [9.9859744e-01 4.5398535e-04 6.7474910e-05 7.6382165e-04 1.1396924e-04
  3.2870801e-06]
 [1.2909678e-04 9.9820876e-01 5.6656206e-04 1.1974232e-04 8.3696672e-05
  8.9223427e-04]
 [9.9917042e-01 3.0851545e-04 5.7565740e-05 3.2091641e-04 1.3895104e-04
  3.6524607e-06]]


In [27]:
input_text = "i remember feeling acutely distressed for a few days"
input_tokens = list(process_text([input_text]))[0]

input_seq = sentence_to_vec(input_tokens, model_cbow)
input_seq = np.expand_dims(input_seq, axis=0)  # (1, 30, 100)

prediction = model.predict(input_seq)
predicted_class = np.argmax(prediction)

print("Predicted class:", predicted_class)
print("Confidence:", np.max(prediction))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step
Predicted class: 4
Confidence: 0.99077165
